# 07 · Phase 6: explainability (Grad-CAM)
Shows **where** the model looks: heatmaps over the face frames and over the spectrogram.
For the fusion model, one backward pass of the *fused* decision explains both modalities at once.

- **Gallery:** for every category, the most confident correct predictions and the most confident mistakes.
- **Single sample:** explain any test clip, as evidence for fake or for real.

**Needs:** a trained checkpoint (notebooks 03–06). A GPU helps but isn't required.

In [ ]:
# --- Session setup: run this first, every time you open the notebook ---
REPO_URL = "https://github.com/tanishq1101/robust-multimodal-deepfake.git"  # "" = use code copied to Drive

import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")

if REPO_URL:
    CODE = "/content/DeepFake"
    if os.path.exists(CODE):
        subprocess.run(["git", "-C", CODE, "pull"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, CODE], check=True)
else:
    # No GitHub repo yet: upload the project folder to Google Drive at MyDrive/deepfake_project/code
    CODE = "/content/drive/MyDrive/deepfake_project/code"
    assert os.path.exists(CODE), f"Upload the project folder to {CODE} or set REPO_URL"

os.chdir(CODE)
!pip install -q -r requirements-colab.txt && pip install -q -e .
sys.path.insert(0, f"{CODE}/src")

from deepfake.config import load_config
cfg = load_config()
print("data_root (Drive):", cfg["data_root"])
print("raw data (VM disk):", cfg["paths"]["raw"])

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import Image, Markdown, display
DATASET = "fakeavceleb"
CKPT = Path(cfg["paths"]["checkpoints"])
# first existing checkpoint in this order
RUN = None
for run in [f"fusion_intermediate_{DATASET}_robust", f"fusion_intermediate_{DATASET}", f"visual_{DATASET}", "visual_dfdc", "audio_asvspoof"]:
    if (CKPT / run / "best.pt").exists():
        RUN = run; break
if RUN is None:
    raise FileNotFoundError("No trained PyTorch checkpoint found. Run notebooks 03–05 first.")
print("explaining:", RUN)
OUT = Path(cfg["paths"]["outputs"]) / f"gallery_{RUN}"

## Gallery

In [ ]:
ds_name = "asvspoof" if RUN.startswith("audio_asvspoof") else ("dfdc" if RUN.endswith("dfdc") else DATASET)
!python -m deepfake.explainability.gallery --checkpoint {CKPT}/{RUN}/best.pt --dataset {ds_name} --per-category 2 --out {OUT}
g = pd.read_csv(OUT / "gallery.csv")
display(g)
for f in g.file[:8]:
    display(Image(str(OUT / f)))

## Explain one clip

In [ ]:
import torch
from deepfake.explainability import explain, render_explanation
from deepfake.explainability.gallery import load_model
from deepfake.training.tasks import TASKS

device = "cuda" if torch.cuda.is_available() else "cpu"
model, rc = load_model(CKPT / RUN / "best.pt", device)
index = Path(cfg["paths"]["processed"]) / ds_name / "index.csv"
ds = TASKS[rc["task"]].build_dataset(index, "test", cfg, False, rc)
print(len(ds), "test clips; categories:", ds.df.get("category", pd.Series()).unique()[:8])

In [ ]:
i = 0                      # change to explore other clips
TARGET = "predicted"       # or "fake" / "real"
batch = {k: v[None].to(device) for k, v in ds[i].items()}
exp = explain(model, rc["task"], batch, target=TARGET)
row = ds.df.iloc[i]
fig = render_explanation(batch, exp, title=f"{row.get('category', '')} · {row.sample_id} · label {int(row.label)}")
plt.show()

**Reading the heatmaps:** red = strong evidence for the explained class. Typical deepfake cues
are blending boundaries around the face, mouth and teeth regions, and eyes. For audio, look at
formant/harmonic structure and high-frequency bands, where vocoders leave artefacts. Grad-CAM is
coarse (7×7 per face, 4×13 per spectrogram), so read it as regions, not pixels. Copy good
examples into `reports/figures/`.